### ML subsurface subsurface predictive modelling
Here we will explore the preformance of ML in predicitng the subsurface structure (tilt) of an eddy from its surface signature

In [1]:
import numpy as np
import matplotlib.pyplot as plt

import seacofs_tilt_tools as tilt

paths = tilt.Paths()
grid = tilt.load_grid(paths.grid, paths.z_r)
df_eddies, df_tilt = tilt.load_tilt_tables(paths)
df_eddies = tilt.add_pv_gradient_terms(df_eddies, grid, core_mean=True)
df_vert = tilt.load_vert(paths)

In [2]:
df_eddies

,Eddy,Day,Cyc,lon,lat,ic,jc,xc,yc,w,...,q22,Rc,psi0,AR,R,Age,Date,fname,TiltDis,TiltDir
0,1,1462,CE,152.716333,-38.866383,171,26,474.560686,127.605656,-0.000032,...,0.737370,29.449052,7.225911,1.973172,20.004291,26,1994-01-02,/srv/scratch/z3533156/26year_BRAN2020/outer_av...,NaN,NaN
1,1,1463,CE,152.836206,-38.814279,173,28,482.821993,136.635735,-0.000041,...,0.816358,29.449052,7.225911,1.859668,18.906549,26,1994-01-03,/srv/scratch/z3533156/26year_BRAN2020/outer_av...,NaN,NaN
2,1,1464,CE,152.905198,-38.816261,175,28,488.606048,138.480541,-0.000035,...,0.967292,32.090160,7.712683,1.745185,20.859920,26,1994-01-04,/srv/scratch/z3533156/26year_BRAN2020/outer_av...,NaN,NaN
3,1,1465,CE,152.719389,-38.902070,171,25,475.960098,123.953440,-0.000020,...,1.051020,34.827275,8.275696,1.433213,18.674495,26,1994-01-05,/srv/scratch/z3533156/26year_BRAN2020/outer_av...,19.823350,87.507772
4,1,1466,CE,152.753007,-38.935135,172,25,479.803789,121.483614,-0.000026,...,1.060328,37.564389,8.838709,1.140614,21.564028,26,1994-01-06,/srv/scratch/z3533156/26year_BRAN2020/outer_av...,15.055658,105.302388
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127421,2982,10646,CE,153.739453,-37.488262,181,61,515.983450,302.961328,-0.000026,...,0.978558,61.312120,19.408085,1.110214,55.409919,28,2019-02-24,/srv/scratch/z3533156/26year_BRAN2020/outer_av...,0.752304,4.278123
127422,2982,10647,CE,153.687115,-37.523299,181,60,512.684226,297.695084,-0.000027,...,0.972594,61.305108,19.388805,1.037331,44.896186,28,2019-02-25,/srv/scratch/z3533156/26year_BRAN2020/outer_av...,1.165623,28.370419
127423,2982,10648,CE,153.699708,-37.537452,181,60,514.199660,296.590511,-0.000021,...,0.949624,61.298095,19.369526,1.055789,38.796956,28,2019-02-26,/srv/scratch/z3533156/26year_BRAN2020/outer_av...,NaN,NaN
127424,2982,10649,CE,153.682657,-37.451789,180,62,510.019363,305.068558,-0.000019,...,0.937880,64.546445,19.604100,1.070344,40.028670,28,2019-02-27,/srv/scratch/z3533156/26year_BRAN2020/outer_av...,NaN,NaN


In [3]:
df_eddies.columns

Index(['Eddy', 'Day', 'Cyc', 'lon', 'lat', 'ic', 'jc', 'xc', 'yc', 'w',
       'Omega', 'q11', 'q12', 'q22', 'Rc', 'psi0', 'AR', 'R', 'Age', 'Date',
       'fname', 'TiltDis', 'TiltDir'],
      dtype='object')

In [5]:
df_vert#['Eddy1']['Day1462']

,Eddy,Day,fnumber,Depth,z,xc,yc,ic,jc,w,q11,q12,q22,Omega0,Omega,Rc,psi0,R
0,1,1462,1461,0.000000,0,474.560686,127.605656,171,26,-0.000032,1.758660,-0.532719,0.737370,NaN,-0.000013,29.449052,7.225911,20.004291
1,1,1462,1461,5.879627,1,476.056668,129.678745,171,26,-0.000030,1.628746,-0.329796,0.680748,-0.000013,-0.000013,35.798824,8.596591,19.126716
2,1,1462,1461,10.725783,2,476.887911,132.134805,172,27,-0.000030,1.593752,-0.319464,0.691486,-0.000013,-0.000014,33.615289,8.107898,21.330116
3,1,1462,1461,16.383097,3,476.324558,136.117738,171,28,-0.000033,1.776776,-0.280829,0.607204,-0.000014,-0.000016,31.928536,8.213464,20.137241
4,1,1462,1461,22.925581,4,478.387952,139.735396,172,28,-0.000034,1.787666,-0.425015,0.660435,-0.000014,-0.000017,30.093898,7.889563,21.334571
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2824604,2982,10650,10641,515.416489,19,508.963912,299.374668,180,61,-0.000017,1.033282,-0.035786,0.969029,-0.000008,-0.000006,75.695297,16.693750,45.049156
2824605,2982,10650,10641,660.806374,20,508.678327,299.023082,180,61,-0.000014,1.045371,-0.055566,0.959552,-0.000007,-0.000005,78.656868,15.933899,46.467550
2824606,2982,10650,10641,858.918478,21,508.485388,298.326056,180,61,-0.000012,1.055097,-0.031142,0.948699,-0.000006,-0.000004,101.415168,18.656178,62.280573
2824607,2982,10650,10641,1128.898331,22,506.706507,297.269787,179,60,-0.000008,0.980631,0.030299,1.020688,-0.000004,-0.000003,115.807964,18.668141,82.943966


In [6]:
df_eddies.columns

Index(['Eddy', 'Day', 'Cyc', 'lon', 'lat', 'ic', 'jc', 'xc', 'yc', 'w',
       'Omega', 'q11', 'q12', 'q22', 'Rc', 'psi0', 'AR', 'R', 'Age', 'Date',
       'fname', 'TiltDis', 'TiltDir'],
      dtype='object')

In [ ]:
'lat', 'w', 'Omega', 'Rc',  'AR', 'Age', 'PV_gradient'